# p086 Partition List 解析笔记

- 题号：p086
- 题目英文名：Partition List
- 题目中文名：分隔链表
- 题目链接：https://leetcode.cn/problems/partition-list/
- 题型：链表 / 指针操作
- 难度：Medium
- 推荐优先级：中
- Java 原代码位置：`solutions-java/leetcode/p086_partition_list/PartitionList.java`


## 1. 题目一句话总结

这道题要求我们把链表按阈值 `x` 分成左右两段：左边都小于 `x`，右边都大于等于 `x`，同时还要保持每一段内部的原有相对顺序。

本质上考的是链表的稳定分组，也就是一边遍历一边把节点按条件拆到两条子链表里，最后再按顺序拼回去。


## 2. 题目理解与约束分析

### 2.1 题目要求

给一条单链表和一个分界值 `x`，我们需要重新组织这条链表，让所有 `< x` 的节点出现在前面，所有 `>= x` 的节点出现在后面。

注意这不是简单排序，因为题目明确要求保持相对顺序。也就是说：

- 小于 `x` 的节点之间，顺序不能乱
- 大于等于 `x` 的节点之间，顺序也不能乱

### 2.2 输入与输出

- 输入：链表头节点 `head`，整数 `x`
- 输出：重组后的链表头节点
- 返回结果含义：返回的新链表满足稳定分区要求

### 2.3 关键约束

- 链表可能为空
- 链表长度可能为 1
- 不能打乱每一部分内部的原始顺序
- 更自然的做法是复用原节点，只重连 `next`

### 2.4 示例分析

输入：`head = [1,4,3,2,5,2]`，`x = 3`

遍历时我们把节点分流到两边：

- `< 3` 的部分：`1 -> 2 -> 2`
- `>= 3` 的部分：`4 -> 3 -> 5`

最后拼接后得到：`[1,2,2,4,3,5]`


## 3. Java 原代码

```java
package leetcode.p086_partition_list;

import util.collection.ListNode;

public class PartitionList {

    public static ListNode partition(ListNode head, int x) {
        if (head == null || head.next == null) {
            return head;
        }

        ListNode leftHead = null;
        ListNode leftTail = null;
        ListNode rightHead = null;
        ListNode rightTail = null;

        while (head != null) {
            if (head.val < x) {
                if (leftHead == null) {
                    leftHead = head;
                } else {
                    leftTail.next = head;
                }
                leftTail = head;
            } else {
                if (rightHead == null) {
                    rightHead = head;
                } else {
                    rightTail.next = head;
                }
                rightTail = head;
            }

            ListNode next = head.next;
            head.next = null;
            head = next;
        }

        if (leftHead == null) {
            return rightHead;
        } else {
            leftTail.next = rightHead;
            return leftHead;
        }
    }
}
```


## 4. 先从 Java 原方案理解

这份 Java 原解法非常值得保留，因为它不是“用额外容器收集后再重建”，而是直接在原链表上做稳定分流。

Java 方案的核心结构有 4 个指针：

- `leftHead` / `leftTail`：维护 `< x` 的链表
- `rightHead` / `rightTail`：维护 `>= x` 的链表

它的处理流程是：

1. 逐个取出原链表当前节点
2. 根据 `head.val < x` 还是 `>= x`，接到左链表或右链表尾部
3. 先保存 `next`，再把当前节点的 `next` 断开
4. 遍历结束后，把左链表尾巴接到右链表头上

这里最关键的细节就是这两句：

```java
ListNode next = head.next;
head.next = null;
```

如果不主动断开，当前节点可能还保留着原链表里的后继关系，后面再拼接时就容易把旧链条错误带进来，甚至形成脏链或环。


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

一种很容易想到但不够好的办法是：

1. 遍历链表，把 `< x` 的值和 `>= x` 的值分别收集到数组
2. 再按数组顺序重新创建新链表

### 5.2 为什么不够好

- 额外用了更多空间
- 做了不必要的节点重建
- 明明原链表节点本身就能复用

### 5.3 优化方向

既然题目只要求重新连接节点顺序，而不是重新生成数据，那么最自然的优化就是沿用 Java 原方案：遍历一次原链表，同时维护两条子链表，最后拼接。

这样能做到：

- 时间复杂度 `O(n)`
- 额外空间 `O(1)`
- 完整保留稳定顺序


## 6. 核心算法讲解

### 6.1 算法名称

- 链表稳定分区
- 链表指针重连

### 6.2 为什么想到这个算法

因为题目要求的是“按条件分组，但组内顺序不能变”。这类要求往往意味着不能排序，而应该按原遍历顺序做稳定收集。

链表场景下，最适合的稳定收集方式就是维护两个尾指针：一个负责收集左边节点，一个负责收集右边节点。

### 6.3 关键状态

- `leftHead`：左链表头节点
- `leftTail`：左链表尾节点
- `rightHead`：右链表头节点
- `rightTail`：右链表尾节点
- `next_node`：保存原链表下一个节点，避免断链后丢失后续部分

### 6.4 步骤拆解

1. 处理空链表和单节点链表
2. 初始化左右两段的头尾指针为 `None`
3. 遍历原链表，每次先判断当前节点属于哪一边
4. 把当前节点接到对应子链表的尾部
5. 保存原来的 `next`，并把当前节点 `next` 断开
6. 继续处理下一个节点
7. 遍历结束后，如果左链表为空，直接返回右链表
8. 否则把 `leftTail.next = rightHead`，返回 `leftHead`


## 7. 过程演示

以 `head = [1,4,3,2,5,2]`，`x = 3` 为例：

```text
初始：
left  = 空
right = 空

读到 1：
1 < 3，接到 left
left  = 1
right = 空

读到 4：
4 >= 3，接到 right
left  = 1
right = 4

读到 3：
3 >= 3，接到 right
left  = 1
right = 4 -> 3

读到 2：
2 < 3，接到 left
left  = 1 -> 2
right = 4 -> 3

读到 5：
5 >= 3，接到 right
left  = 1 -> 2
right = 4 -> 3 -> 5

读到 2：
2 < 3，接到 left
left  = 1 -> 2 -> 2
right = 4 -> 3 -> 5

最后拼接：
1 -> 2 -> 2 -> 4 -> 3 -> 5
```

从这个过程里最容易漏掉的不是分类，而是“每次都要把当前节点从原链表里摘干净”。这正是 Java 原代码里 `head.next = null` 的价值。


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional


@dataclass
class ListNode:
    val: int = 0
    next: Optional["ListNode"] = None


def build_linked_list(values: list[int]) -> Optional[ListNode]:
    dummy = ListNode()
    cur = dummy
    for value in values:
        cur.next = ListNode(value)
        cur = cur.next
    return dummy.next


def linked_list_to_list(head: Optional[ListNode]) -> list[int]:
    ans = []
    while head is not None:
        ans.append(head.val)
        head = head.next
    return ans


In [ ]:
class Solution:
    def partition(self, head: Optional[ListNode], x: int) -> Optional[ListNode]:
        # 这里刻意贴近 Java 原实现，保留 left/right 两段的 head 和 tail。
        if head is None or head.next is None:
            return head

        left_head = None
        left_tail = None
        right_head = None
        right_tail = None

        while head is not None:
            next_node = head.next
            # 对照 Java 的 head.next = null，先把当前节点从原链表摘出来。
            head.next = None

            if head.val < x:
                if left_head is None:
                    left_head = head
                else:
                    left_tail.next = head
                left_tail = head
            else:
                if right_head is None:
                    right_head = head
                else:
                    right_tail.next = head
                right_tail = head

            head = next_node

        if left_head is None:
            return right_head

        left_tail.next = right_head
        return left_head


## 8. 代码逐段讲解

### 8.1 初始化部分

这里没有用虚拟头节点，而是故意沿用 Java 的四指针写法。这样和原代码一一对应，更容易做 Java 到 Python 的迁移理解。

### 8.2 主循环

每一轮循环都做三件事：

1. 先保存 `next_node`
2. 把 `head.next` 断开
3. 根据大小接入左链表或右链表

这样做的重点是：当前节点一旦被分类，它就已经从原链表里独立出来了，后续拼接不会被旧指针干扰。

### 8.3 更新答案

实际上答案不是一次性生成的，而是在遍历过程中逐步维护：

- 左部分不断通过 `left_tail` 追加
- 右部分不断通过 `right_tail` 追加

最终的“答案生成”动作只有一步：`left_tail.next = right_head`

### 8.4 返回结果

如果左边为空，说明所有节点都 `>= x`，那直接返回右链表头即可；否则返回拼接后的左链表头。


## 9. 复杂度分析

- 时间复杂度：`O(n)`
- 为什么是这个时间复杂度：每个节点只遍历一次，只做常数次指针操作
- 空间复杂度：`O(1)`
- 为什么是这个空间复杂度：只用了若干个指针变量，没有额外数组或哈希表


## 10. 易错点

1. 忘记先保存 `next_node = head.next`，结果一断链就找不到后面的节点了。
2. 忘记执行 `head.next = None`，导致旧链关系残留，拼接后结构出错。
3. 左链表或右链表第一次插入节点时，没有单独初始化头指针。
4. 最后直接返回 `left_head`，却忘了把 `left_tail.next` 接到 `right_head`。
5. 没处理 `left_head is None` 的情况，导致全都在右边时返回错误。


## 11. 面试时怎么讲

可以这样概括：

> 这题我不会去排序，因为题目要求保持相对顺序，本质上是稳定分区。
> 我会沿着原链表遍历一遍，维护两条子链表：一条放 `< x` 的节点，一条放 `>= x` 的节点。
> 每拿到一个节点，我会先保存它的下一个节点，再把当前节点 `next` 断开，然后接到对应子链表尾部。
> 最后把左链表尾巴接到右链表头上即可。
> 这样时间复杂度是 `O(n)`，额外空间复杂度是 `O(1)`。


## 12. 实际应用场景

这题可以对应到“按规则稳定分流任务，但不能改变同类任务原始先后顺序”的业务场景。

### 具体业务案例：消息任务稳定分流

#### 业务背景

一个消息处理系统里，任务已经按到达时间形成队列。现在要把优先级低于阈值的任务先送去普通通道，把高优先级任务送去另一条通道。

#### 输入是什么

输入是一条按到达时间串起来的任务链表，每个节点有一个优先级字段。

#### 算法介入点

系统需要在不打乱同类任务先后顺序的前提下，按阈值把任务拆成两段。

#### 输出是什么

输出是一个重新连接后的任务队列，前半段是低于阈值的任务，后半段是高于等于阈值的任务。

#### 价值是什么

它解决的是稳定分流问题：既满足分类要求，又不破坏原始到达顺序，适合后续处理链继续消费。


In [ ]:
head = build_linked_list([1, 4, 3, 2, 5, 2])
result = Solution().partition(head, 3)
print('示例一:', linked_list_to_list(result))

head = build_linked_list([2, 1])
result = Solution().partition(head, 2)
print('边界示例:', linked_list_to_list(result))

head = build_linked_list([4, 5, 6])
result = Solution().partition(head, 3)
print('左侧为空:', linked_list_to_list(result))


## 13. Demo 输出说明

- 第一组输出应为 `[1, 2, 2, 4, 3, 5]`，说明主流程正确。
- 第二组输出 `[1, 2]`，说明当小于 `x` 的节点在后面出现时，仍然能被稳定挪到前段。
- 第三组输出 `[4, 5, 6]`，说明当左链表为空时，代码能正确直接返回右链表。


## 14. 一句话复盘

> 这题最关键的不是“分成两段”，而是像 Java 原解那样在遍历时把节点先摘干净、再稳定接回去。
